# COVID-19 EDA — covid_19_clean_complete.csv
**024BSCIT043 — Sulav Pandey — Section A**

This notebook follows the same structure as the sample EDA (data.csv / cars),
but a few steps are adapted because this dataset has a **time-series structure**:
each row is a country/province on a specific date, and Confirmed/Deaths/Recovered
are *cumulative* counts, not one-off measurements. Notes are added wherever the
approach differs from the sample, and why.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns                       #visualisation
import matplotlib.pyplot as plt             #visualisation
%matplotlib inline     
sns.set(color_codes=True)

In [3]:
df = pd.read_csv('covid_19_clean_complete.csv')
df.head(5)

,Province/State,Country/Region,Lat,Long,Date,Confirmed,Deaths,Recovered,Active,WHO Region
0,NaN,Afghanistan,33.93911,67.709953,2020-01-22,0,0,0,0,Eastern Mediterranean
1,NaN,Albania,41.15330,20.168300,2020-01-22,0,0,0,0,Europe
2,NaN,Algeria,28.03390,1.659600,2020-01-22,0,0,0,0,Africa
3,NaN,Andorra,42.50630,1.521800,2020-01-22,0,0,0,0,Europe
4,NaN,Angola,-11.20270,17.873900,2020-01-22,0,0,0,0,Africa


In [4]:
df.tail(5)                        # To display the bottom 5 rows

,Province/State,Country/Region,Lat,Long,Date,Confirmed,Deaths,Recovered,Active,WHO Region
49063,NaN,Sao Tome and Principe,0.186400,6.613100,2020-07-27,865,14,734,117,Africa
49064,NaN,Yemen,15.552727,48.516388,2020-07-27,1691,483,833,375,Eastern Mediterranean
49065,NaN,Comoros,-11.645500,43.333300,2020-07-27,354,7,328,19,Africa
49066,NaN,Tajikistan,38.861000,71.276100,2020-07-27,7235,60,6028,1147,Europe
49067,NaN,Lesotho,-29.610000,28.233600,2020-07-27,505,12,128,365,Africa


Checking the types of data

In [ ]:
df.dtypes

**Note:** `Date` is currently an `object` (string), not a real date.
For a time-series dataset this matters a lot — we can't sort, plot over time,
or extract month/week from a string properly. So unlike the cars sample
(which has no date column), we convert it here before doing anything else.

In [ ]:
# Convert Date column to actual datetime type
df['Date'] = pd.to_datetime(df['Date'])
df.dtypes

In [ ]:
#Dropping irrelevant columns
df = df.drop(['WHO Region'], axis=1)
df.head(5)

In [ ]:
#Renaming the columns
df = df.rename(columns={"Country/Region": "Country", "Province/State": "Province"})
df.head(5)

In [ ]:
#Dropping the duplicate rows
df.shape

In [ ]:
duplicate_rows_df = df[df.duplicated()]
print("number of duplicate rows: ", duplicate_rows_df.shape)

In [ ]:
df.count()      # Used to count the number of rows

In [ ]:
df = df.drop_duplicates()
df.head(5)

In [ ]:
df.count()

In [ ]:
#Dropping the missing or null values.
print(df.isnull().sum())

In [ ]:
# Instead of dropna() (which would wipe out most rows), fill missing
# Province values with a placeholder since the country-level data is still valid
df['Province'] = df['Province'].fillna('Not Reported')

print(df.isnull().sum())   # Province should now show 0 missing

In [ ]:
# Plotting a basic histogram
plt.hist(x=df['Confirmed'], bins=30, color='skyblue', edgecolor='black')
 
# Adding labels and title
plt.xlabel('Confirmed Cases')
plt.ylabel('Frequency')
plt.title('Basic Histogram of Confirmed Cases (all rows, all dates)')
 
# Display the plot
plt.show()

In [ ]:
#Plot different features against one another (scatter), against frequency (histogram)
df.Country.value_counts().nlargest(40).plot(kind='bar', figsize=(10,5))
plt.title("Number of records by country")
plt.ylabel('Number of records (rows)')
plt.xlabel('Country');

Notice every country has roughly the *same* number of records — that's
expected, since each country gets one row per date over the same date range.
This bar chart is really confirming the dataset's structure rather than
showing variation, which is a different takeaway than the cars version
(where it showed which manufacturers are more common).

## Extra step: Global trend over time

The cars sample has no equivalent to this, but since our dataset is
time-series in nature, the single most informative EDA step is plotting
totals over time — this is the visualization that actually shows the
"story" of the pandemic, which a histogram or single boxplot can't.

In [ ]:
# Aggregate worldwide totals for each date
global_trend = df.groupby('Date')[['Confirmed', 'Deaths', 'Recovered']].sum()

plt.figure(figsize=(10,6))
plt.plot(global_trend.index, global_trend['Confirmed'], label='Confirmed')
plt.plot(global_trend.index, global_trend['Deaths'], label='Deaths')
plt.plot(global_trend.index, global_trend['Recovered'], label='Recovered')
plt.xlabel('Date')
plt.ylabel('Count')
plt.title('Global COVID-19 Trend Over Time')
plt.legend()
plt.show()

## Detecting Outliers

**Adapted step:** Running a boxplot on `Confirmed` across *all* rows (all dates)
would mostly show the early-pandemic near-zero values as a dense cluster and
later huge values as "outliers" — which isn't really what outlier detection
is meant to find here. Instead, we take a **snapshot of the latest date only**,
so each country contributes exactly one value, and outliers genuinely mean
"countries with unusually high case counts compared to the rest".

In [ ]:
# Take only the most recent date so each country contributes one row
latest_date = df['Date'].max()
latest_df = df[df['Date'] == latest_date]
print(f"Snapshot date: {latest_date}")
latest_df.shape

In [ ]:
#Detecting Outliers
sns.boxplot(x=latest_df['Confirmed'])

In [ ]:
sns.boxplot(x=latest_df['Deaths'])

In [ ]:
sns.boxplot(x=latest_df['Recovered'])

The countries appearing as individual dots far to the right of the box
are the outliers — these are countries with case counts far above the
typical (median) country, e.g. the US, India, Brazil. This matches what we'd
expect from real-world COVID spread, which is a good sanity check that the
data is clean.

For Correlation Matrix just use numerical value column as selected below

In [ ]:
selected_columns = ['Confirmed', 'Deaths', 'Recovered', 'Active']
df_selected = latest_df[selected_columns]


plt.figure(figsize=(15, 10))

# Using Seaborn to create a heatmap
sns.heatmap(df_selected.corr(), annot=True, fmt='.2f', cmap='Pastel2', linewidths=2)

plt.title('Correlation Heatmap (latest snapshot)')
plt.show()

**Note:** We use `latest_df` (one row per country) here instead of the
full `df` (all dates). Correlating cumulative totals across all dates would
be misleading — Confirmed and Deaths would look almost perfectly correlated
simply because both grow over time for every country, regardless of any real
relationship between them. Using one snapshot avoids that trap.

In [ ]:
#scatter plot
fig, ax = plt.subplots(figsize=(10,6))
ax.scatter(latest_df['Confirmed'], latest_df['Deaths'])
ax.set_xlabel('Confirmed')
ax.set_ylabel('Deaths')
ax.set_title('Confirmed vs Deaths (latest snapshot, one point per country)')
plt.show()